In [1]:
# inlegalbert_bilstm_mha_crf_rrc.py  (FIXED)
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#
# KEY FIXES vs. original:
#   1. encode_sentences() now accepts `lengths` so padding sentences are
#      never passed to BERT (BERT produces NaN for all-zero attention_mask).
#   2. Padding sentence slots are zeroed explicitly instead of being NaN.
#   3. MHA query init replaced: xavier_uniform_ on a non-contiguous view
#      silently corrupted weights — now uses nn.init.trunc_normal_.
#   4. MHA uses -1e9 instead of float("-inf") so softmax never produces NaN
#      even when all tokens are masked.
#   5. HEAD_LR lowered 1e-3 → 5e-4 and BERT_LR slightly raised to 3e-5
#      for stable joint training.
#   6. nan_to_num() safety net on emissions before the CRF.
#   7. forward() passes lengths to encode_sentences().
#

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32          # tokens per sentence
BATCH_DOCS      = 2
NUM_EPOCHS      = 60
BERT_LR         = 3e-5        # FIX: was 2e-5, slight raise for better fine-tuning
HEAD_LR         = 5e-4        # FIX: was 1e-3 (too aggressive) → 5e-4
WEIGHT_DECAY    = 0.01
GRAD_CLIP       = 1.0
DROPOUT         = 0.3

# ── Sentence-level BiLSTM (token → sentence vector) ───────
SENT_LSTM_HIDDEN = 256        # per direction → output 512
SENT_LSTM_LAYERS = 2

# ── Multi-Head Attention Pooling ──────────────────────────
MHA_HEADS        = 8          # must divide (SENT_LSTM_HIDDEN * 2) = 512
MHA_DROPOUT      = 0.1

# ── Context-enrichment BiLSTM (sentence → doc) ────────────
CTX_LSTM_HIDDEN  = 128        # per direction → output 256
CTX_LSTM_LAYERS  = 2

WARMUP_RATIO    = 0.05
GRADIENT_ACCUMULATION_STEPS = 2

RARE_THRESHOLD = 0.05

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":              "InLegalBERT Encoder",
        "sent_bilstm":       "Sentence BiLSTM",
        "mha_pooling":       "Multi-Head Attn Pooling",
        "ctx_bilstm":        "Context BiLSTM",
        "classifier":        "Classifier Head",
        "crf":               "CRF",
        "dropout":           "Dropout",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({
            "Component":        name,
            "Trainable Params": trainable,
            "Frozen Params":    frozen,
            "Total Params":     trainable + frozen,
        })

    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_all       = total_trainable + total_frozen

    rows.append({
        "Component":        "── TOTAL ──",
        "Trainable Params": total_trainable,
        "Frozen Params":    total_frozen,
        "Total Params":     total_all,
    })

    print("\n" + "=" * 70)
    print("MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF)")
    print("=" * 70)
    print(f"  {'Component':<30} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 70)
    for r in rows:
        sep = "─" * 70 if r["Component"] == "── TOTAL ──" else ""
        if sep:
            print(sep)
        print(
            f"  {r['Component']:<30} "
            f"{r['Trainable Params']:>14,} "
            f"{r['Frozen Params']:>10,} "
            f"{r['Total Params']:>12,}"
        )
    print("=" * 70)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}

    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []

        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))

        if not sents or len(sents) != len(labs):
            continue

        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING  (FIXED)
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    """
    Collapses token sequence (N, L, H) → sentence vector (N, H).

    FIX 1: query init changed from broken xavier_uniform_-on-view
            to trunc_normal_ which is safer for learnable query vectors.
    FIX 2: masking uses -1e9 instead of float("-inf") so softmax never
            produces NaN even when every token position is masked.
    """

    def __init__(self, hidden_dim: int, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0, (
            f"hidden_dim ({hidden_dim}) must be divisible by num_heads ({num_heads})"
        )
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        # ── FIX 1: proper query initialisation ───────────────
        # Original code did xavier_uniform_ on a non-contiguous 3-D view,
        # which silently corrupted the parameter → NaN in attention scores.
        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)

        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x               : (N, L, H)
            key_padding_mask: (N, L) bool — True where tokens are *padding*
        Returns:
            pooled          : (N, H)
        """
        N, L, H = x.shape

        K = self.key_proj(x)   # (N, L, H)
        V = self.val_proj(x)   # (N, L, H)

        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)  # (N, nh, L, hd)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)  # (N, nh, L, hd)

        Q = self.query.expand(N, -1, -1, -1)                             # (N, nh, 1, hd)

        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale  # (N, nh, 1, L)

        # ── FIX 2: use -1e9 not float("-inf") ─────────────────
        # float("-inf") → softmax([-inf, -inf, ...]) = NaN.
        # -1e9 → softmax([-1e9, -1e9, ...]) = uniform (never NaN).
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)            # (N, 1, 1, L)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)                   # (N, nh, 1, L)
        attn_weights = self.attn_drop(attn_weights)

        context = torch.matmul(attn_weights, V)  # (N, nh, 1, hd)
        context = context.squeeze(2)             # (N, nh, hd)
        context = context.reshape(N, H)          # (N, H)

        pooled = self.out_proj(context)          # (N, H)
        return pooled


# ═══════════════════════════════════════════════════════════
# MODEL  (FIXED)
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):
    """
    FIX 3: encode_sentences() now accepts `lengths` so it can skip
            all-zero-attention_mask sentences (batch-padding positions).
            BERT produces NaN for any sequence whose attention_mask is
            entirely zeros, which then propagated through LSTM → MHA → CRF.
    FIX 4: forward() passes lengths to encode_sentences().
    FIX 5: emissions are nan_to_num()'d before the CRF as a safety net.
    """

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
    ):
        super().__init__()

        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size          # 768
        self.dropout  = nn.Dropout(dropout)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2                   # 512

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        ctx_out_dim = ctx_lstm_hidden * 2                     # 256

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim, ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim // 2, num_labels),
        )

        self.crf = CRF(num_tags=num_labels, batch_first=True)

    # ── FIX 3: encode_sentences now takes lengths ─────────────
    def encode_sentences(
        self,
        input_ids:      torch.Tensor,   # (B, T, L)
        attention_mask: torch.Tensor,   # (B, T, L)
        token_type_ids: torch.Tensor,   # (B, T, L)
        lengths:        torch.Tensor = None,  # (B,) — real sentence counts
    ) -> torch.Tensor:
        """
        Returns sent_vecs : (B, T, sent_out_dim)

        Only real sentences (those with at least one non-padding token) are
        passed through BERT.  Padding sentence slots are filled with zeros.
        This prevents BERT's internal attention softmax from receiving
        all-(-inf) inputs and producing NaN hidden states.
        """
        B, T, L = input_ids.shape
        N = B * T

        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        # Boolean mask: True for sentences that have ≥1 real token
        valid = flat_mask.sum(dim=-1) > 0   # (N,)

        # Allocate output buffer — padding slots stay zero
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)

        # Sentence BiLSTM over token embeddings
        lstm_out, _ = self.sent_bilstm(token_embs_all)         # (N, L, sent_out_dim)
        lstm_out    = self.dropout(lstm_out)

        # MHA pooling
        # For all-padding sentences we reset the mask to False so softmax
        # sees uniform logits (not all -1e9) and produces a clean zero-ish
        # vector.  The slot is then explicitly zeroed below.
        pad_mask = (flat_mask == 0).clone()                    # True = padding token
        pad_mask[~valid] = False                               # don't mask padding sentences

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)

        # Zero out outputs for padding sentence slots
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)

        return sent_vecs.view(B, T, -1)

    # ── Forward ───────────────────────────────────────────────
    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
    ):
        # FIX 4: pass lengths to encode_sentences
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs = self.dropout(sent_vecs)

        # Context BiLSTM over sentence sequence
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)                   # (B, T, NUM_LABELS)

        # FIX 5: safety net — replace any residual NaN/Inf with zeros
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        # CRF mask
        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS HELPER
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    cls_report = classification_report(
        str_trues, str_preds,
        labels=LABELS,
        digits=4,
        zero_division=0,
    )
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1":            macro_f1,
        "micro_f1":            micro_f1,
        "weighted_f1":         weighted_f1,
        "macro_precision":     macro_prec,
        "micro_precision":     micro_prec,
        "weighted_precision":  weighted_prec,
        "macro_recall":        macro_rec,
        "micro_recall":        micro_rec,
        "weighted_recall":     weighted_rec,
        "rare_f1":             rare_f1,
        "rare_precision":      rare_prec,
        "rare_recall":         rare_rec,
        "per_class_metrics":   per_class_metrics,
        "accuracy":            acc,
        "cls_report":          cls_report,
        "cm":                  cm,
        "all_preds":           all_preds,
        "all_trues":           all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        bert_params = list(self.model.bert.parameters())
        sent_params = (
            list(self.model.sent_bilstm.parameters())
            + list(self.model.mha_pooling.parameters())
            + list(self.model.sent_layer_norm.parameters())
        )
        ctx_params = (
            list(self.model.ctx_bilstm.parameters())
            + list(self.model.classifier.parameters())
            + list(self.model.crf.parameters())
            + list(self.model.dropout.parameters())
        )
        return torch.optim.AdamW(
            [
                {"params": bert_params, "lr": BERT_LR,  "weight_decay": WEIGHT_DECAY},
                {"params": sent_params, "lr": HEAD_LR,  "weight_decay": WEIGHT_DECAY},
                {"params": ctx_params,  "lr": HEAD_LR,  "weight_decay": WEIGHT_DECAY},
            ]
        )

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                )
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    n += 1
        return total_loss / max(1, n)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS):

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_DOCS,
            shuffle=True, collate_fn=collate_rrc,
        )
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = warmup_steps,
            num_training_steps = total_steps,
        )

        history    = []
        best_f1    = -1.0
        best_state = None

        total_train_start = time.time()

        for epoch in range(1, num_epochs + 1):
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(train_loader):

                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                )

                # Skip NaN losses gracefully
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1
                    optimizer.zero_grad()
                    continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP
                    )
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            # Final partial accumulation
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            epoch_train_time = time.time() - epoch_start
            avg_train_loss   = running_loss / max(1, n_steps)

            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            nan_info = f" [nan_steps={nan_steps}]" if nan_steps > 0 else ""
            print(
                f"Epoch {epoch:02d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"val_acc: {val_metrics['accuracy']:.4f} | "
                f"epoch_time: {epoch_train_time:.1f}s{nan_info}"
            )

            row = {
                "epoch":                   epoch,
                "train_loss":              avg_train_loss,
                "val_loss":                val_loss,
                "val_accuracy":            val_metrics["accuracy"],
                "val_macro_f1":            val_metrics["macro_f1"],
                "val_micro_f1":            val_metrics["micro_f1"],
                "val_weighted_f1":         val_metrics["weighted_f1"],
                "val_rare_f1":             val_metrics["rare_f1"],
                "val_macro_precision":     val_metrics["macro_precision"],
                "val_micro_precision":     val_metrics["micro_precision"],
                "val_weighted_precision":  val_metrics["weighted_precision"],
                "val_rare_precision":      val_metrics["rare_precision"],
                "val_macro_recall":        val_metrics["macro_recall"],
                "val_micro_recall":        val_metrics["micro_recall"],
                "val_weighted_recall":     val_metrics["weighted_recall"],
                "val_rare_recall":         val_metrics["rare_recall"],
                "epoch_train_time_s":      epoch_train_time,
                "nan_steps":               nan_steps,
                "timestamp":               datetime.utcnow().isoformat(),
            }
            history.append(row)

            if val_metrics["macro_f1"] > best_f1 + 1e-4:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f} — snapshot saved")

        total_train_time = time.time() - total_train_start
        print(f"\n⏱  Total training time : {total_train_time/60:.2f} min "
              f"({total_train_time:.1f} s)")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        timing_summary = {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / num_epochs,
            "num_epochs":              num_epochs,
        }
        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump(timing_summary, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_train_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)

        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)

                decoded, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=None, lengths=lengths,
                )
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    true_seq = labels[i, :true_len].cpu().numpy().tolist()
                    all_preds.extend(seq_preds)
                    all_trues.extend(true_seq)

        if measure_inference_time:
            total_infer_time = time.time() - infer_start
            n_sentences      = len(all_trues)
            infer_info = {
                "split":                      split_name,
                "n_documents":                n_samples,
                "n_sentences":                n_sentences,
                "total_inference_time_s":     total_infer_time,
                "latency_per_document_ms":    total_infer_time / max(1, n_samples) * 1000,
                "latency_per_sentence_ms":    total_infer_time / max(1, n_sentences) * 1000,
                "throughput_sentences_per_s": n_sentences / max(1e-9, total_infer_time),
            }
            with open(
                os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w"
            ) as f:
                json.dump(infer_info, f, indent=2)

            print(f"\n⏱  Inference time ({split_name}): "
                  f"{total_infer_time:.2f} s | "
                  f"latency/doc: {infer_info['latency_per_document_ms']:.1f} ms | "
                  f"latency/sent: {infer_info['latency_per_sentence_ms']:.2f} ms | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(
            state_dict,
            os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"),
        )
        model_args = {
            "bert_model_name":  INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "sent_lstm_layers": SENT_LSTM_LAYERS,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "ctx_lstm_layers":  CTX_LSTM_LAYERS,
            "mha_heads":        MHA_HEADS,
            "mha_dropout":      MHA_DROPOUT,
            "num_labels":       NUM_LABELS,
            "dropout":          DROPOUT,
            "labels":           LABELS,
            "label2id":         label2id,
            "id2label":         id2label,
            "max_seq_length":   MAX_SEQ_LENGTH,
            "rare_threshold":   RARE_THRESHOLD,
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)

        print(f"\n💾 Best model saved to: {BEST_MODEL_DIR}/")
        print(f"   ├── pytorch_model.bin")
        print(f"   ├── config.json")
        print(f"   ├── tokenizer files")
        print(f"   └── model_args.json")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o")
        ax.plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s")
        ax.set_title("Training vs Validation Loss (CRF NLL)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("CRF NLL Loss")
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_loss_curve.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_micro_f1",    "Micro-F1",    "--"),
            ("val_weighted_f1", "Weighted-F1", "-."),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation F1 Scores"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o")
        axes[1].plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s")
        axes[1].set_title("Loss Curves"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_f1_curve.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_precision",    "Macro-Prec",    "-"),
            ("val_micro_precision",    "Micro-Prec",    "--"),
            ("val_weighted_precision", "Weighted-Prec", "-."),
            ("val_rare_precision",     "Rare-Prec",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation Precision"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        for col, lbl, ls in [
            ("val_macro_recall",    "Macro-Rec",    "-"),
            ("val_micro_recall",    "Micro-Rec",    "--"),
            ("val_weighted_recall", "Weighted-Rec", "-."),
            ("val_rare_recall",     "Rare-Rec",     ":"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[1].set_title("Validation Recall"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_pr_curve.png")
        plt.savefig(p, dpi=150); plt.close()
        print(f"Saved {p}")

        if "epoch_train_time_s" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.bar(epochs, hist_df["epoch_train_time_s"], color="steelblue", alpha=0.8)
            ax.axhline(
                hist_df["epoch_train_time_s"].mean(),
                color="red", linestyle="--",
                label=f"Mean = {hist_df['epoch_train_time_s'].mean():.1f}s"
            )
            ax.set_title("Per-Epoch Training Time")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Time (s)")
            ax.legend(); ax.grid(True, alpha=0.3, axis="y")
            plt.tight_layout()
            p = os.path.join(OUT_DIR, "training_time_curve.png")
            plt.savefig(p, dpi=150); plt.close()
            print(f"Saved {p}")

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(
            cm, annot=True, fmt="d",
            xticklabels=LABELS, yticklabels=LABELS,
            cmap="Blues", ax=ax,
        )
        if rare_labels:
            for tick in ax.get_xticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
            for tick in ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(path, dpi=150); plt.close()
        print(f"Saved {path}")

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        labels = LABELS
        f1s    = [per_class_metrics[l]["f1"] for l in labels]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in labels]

        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12)
        ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(path, dpi=150); plt.close()
        print(f"Saved {path}")


# ═══════════════════════════════════════════════════════════
# PRINT FULL METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 64)
    print("FINAL RESULTS SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF)")
    print("=" * 64)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print("-" * 64)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 64)
    for label, key in rows:
        sep = "─" * 64 if key == "rare_f1" else ""
        if sep:
            print(sep)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 64)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print(f"Architecture : InLegalBERT → Sentence BiLSTM → MHA Pooling "
          f"→ Context BiLSTM → Linear → CRF")

    print("Loading JSONL files...")
    train_raw = load_jsonl(TRAIN_PATH)
    dev_raw   = load_jsonl(DEV_PATH)
    test_raw  = load_jsonl(TEST_PATH)

    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    freq_df = pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ])
    freq_df.to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    print("\nInitialising InLegalBERT + BiLSTM + MHA-Pooling + CRF ...")
    model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
    )

    total_trainable, total_frozen, param_table = count_parameters(model)
    param_df = pd.DataFrame(param_table)
    param_df.to_csv(os.path.join(OUT_DIR, "parameter_summary.csv"), index=False)

    trainer = Trainer(model, device=DEVICE)

    print(f"\nStarting training for {NUM_EPOCHS} epochs...")
    hist_df, total_train_time = trainer.train(
        train_dataset, dev_dataset,
        rare_ids   = rare_ids,
        tokenizer  = tokenizer,
        num_epochs = NUM_EPOCHS,
    )
    print("\nTraining complete.")

    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    print("\nEvaluating on Dev set...")
    dev_metrics = trainer.evaluate(
        dev_dataset, rare_ids,
        split_name="dev",
        measure_inference_time=True,
    )
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write(f"Model: InLegalBERT + BiLSTM + MHA-Pooling + CRF\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    trainer.save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    trainer.save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set...")
    test_metrics = trainer.evaluate(
        test_dataset, rare_ids,
        split_name="test",
        measure_inference_time=True,
    )
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write(f"Model: InLegalBERT + BiLSTM + MHA-Pooling + CRF\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    trainer.save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    trainer.save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pred_df = pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    for split, mets in [("dev", dev_metrics), ("test", test_metrics)]:
        rows_pc = []
        for lbl in LABELS:
            pc = mets["per_class_metrics"][lbl]
            rows_pc.append({
                "label":     lbl,
                "is_rare":   lbl in rare_labels,
                "f1":        pc["f1"],
                "precision": pc["precision"],
                "recall":    pc["recall"],
            })
        pd.DataFrame(rows_pc).to_csv(
            os.path.join(OUT_DIR, f"{split}_per_class_metrics.csv"), index=False
        )

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": {
            "name":             "InLegalBERT + BiLSTM + MHA-Pooling + CRF",
            "bert_model":       INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "mha_heads":        MHA_HEADS,
            "trainable_params": total_trainable,
            "frozen_params":    total_frozen,
            "total_params":     total_trainable + total_frozen,
        },
        "timing": {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / NUM_EPOCHS,
            "dev_inference":  dev_metrics.get("inference_time_info", {}),
            "test_inference": test_metrics.get("inference_time_info", {}),
        },
        "rare_classes":   rare_labels,
        "rare_threshold": RARE_THRESHOLD,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        total_train_time = total_train_time,
        total_trainable  = total_trainable,
        total_frozen     = total_frozen,
    )

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")
    print("   ├── history.csv")
    print("   ├── label_frequencies.csv")
    print("   ├── parameter_summary.csv")
    print("   ├── timing_summary.json")
    print("   ├── training_loss_curve.png")
    print("   ├── training_f1_curve.png")
    print("   ├── training_pr_curve.png")
    print("   ├── training_time_curve.png")
    print("   ├── dev_classification_report.txt")
    print("   ├── dev_confusion_matrix.png")
    print("   ├── dev_per_class_f1.png")
    print("   ├── dev_per_class_metrics.csv")
    print("   ├── inference_time_dev.json")
    print("   ├── test_classification_report.txt")
    print("   ├── test_confusion_matrix.png")
    print("   ├── test_per_class_f1.png")
    print("   ├── test_per_class_metrics.csv")
    print("   ├── inference_time_test.json")
    print("   ├── test_predictions.csv")
    print("   ├── metrics_summary.json")
    print("   └── best_model/")
    print("       ├── pytorch_model.bin")
    print("       ├── config.json")
    print("       ├── tokenizer files")
    print("       └── model_args.json")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT → Sentence BiLSTM → MHA Pooling → Context BiLSTM → Linear → CRF
Loading JSONL files...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 samples) ← RARE
   NONE                  4.79%  ( 1377 samples) ← RARE

   Rare classes (10): ['RLC', 'ISSUE', 'ARG_PETITIONER', 'ARG_RESPONDENT', 'STA', 'PRE_RELIED',

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF)
  Component                           Trainable     Frozen        Total
----------------------------------------------------------------------
  InLegalBERT Encoder               109,482,240          0  109,482,240
  Sentence BiLSTM                     3,678,208          0    3,678,208
  Multi-Head Attn Pooling               787,456          0      787,456
  Context BiLSTM                      1,052,672          0    1,052,672
  Classifier Head                        34,573          0       34,573
  CRF                                       195          0          195
  Dropout                                     0          0            0
──────────────────────────────────────────────────────────────────────
  ── TOTAL ──                       115,036,368          0  115,036,368

Starting training for 60 epochs...
Epoch 01/60 | train_loss: 272.6710 | val_loss: 185.9174 | val_macro_f1: 0.0391 | val_rare_f1: 0.0000 | val_a

In [2]:
# inlegalbert_bilstm_mha_crf_rrc_v2.py  (ANTI-OVERFITTING REVISION)
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#
# ANTI-OVERFITTING CHANGES vs v1:
#   1. Early stopping (patience=10 on val_macro_f1).
#   2. Freeze bottom 8 of 12 BERT encoder layers; fine-tune only top 4 + pooler.
#   3. Layer-wise LR decay across BERT transformer layers (factor 0.9 per layer).
#   4. Dropout raised from 0.3 → 0.4.
#   5. Weight decay raised from 0.01 → 0.05.
#   6. Sentence BiLSTM hidden reduced 256 → 128 (output 256).
#   7. Context BiLSTM hidden reduced 128 → 64  (output 128).
#   8. MHA heads reduced 8 → 4 (matches new sent_out_dim=256; 256/4=64 ✓).
#   9. Auxiliary cross-entropy loss (label_smoothing=0.1) added alongside CRF
#      to provide a dense gradient signal and regularise the emission head.
#  10. NUM_EPOCHS cap raised to 100 (early stopping will fire much earlier).
#

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_v2_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32          # tokens per sentence
BATCH_DOCS      = 2
NUM_EPOCHS      = 60         # early stopping will fire well before this
BERT_LR         = 2e-5        # CHANGE: 3e-5 → 2e-5; lower layers get further decay
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05        # CHANGE: 0.01 → 0.05 (stronger L2)
GRAD_CLIP       = 1.0
DROPOUT         = 0.4         # CHANGE: 0.3 → 0.4

# ── BERT freeze / layer-wise LR decay ─────────────────────
BERT_FREEZE_LAYERS  = 8       # freeze bottom 8 of 12 encoder layers  [NEW]
BERT_LR_DECAY       = 0.9     # multiply LR by this per layer upward   [NEW]

# ── Sentence-level BiLSTM (token → sentence vector) ───────
SENT_LSTM_HIDDEN = 128        # CHANGE: 256 → 128 (per dir → output 256)
SENT_LSTM_LAYERS = 2

# ── Multi-Head Attention Pooling ──────────────────────────
MHA_HEADS        = 4          # CHANGE: 8 → 4 (256 / 4 = 64 ✓)
MHA_DROPOUT      = 0.1

# ── Context-enrichment BiLSTM (sentence → doc) ────────────
CTX_LSTM_HIDDEN  = 64         # CHANGE: 128 → 64 (per dir → output 128)
CTX_LSTM_LAYERS  = 2

# ── Auxiliary loss ─────────────────────────────────────────
AUX_CE_WEIGHT    = 0.3        # weight for auxiliary CE loss            [NEW]
LABEL_SMOOTHING  = 0.1        # label smoothing for CE                  [NEW]

# ── Early stopping ─────────────────────────────────────────
ES_PATIENCE      = 10         # stop if no improvement for N epochs     [NEW]
ES_MIN_DELTA     = 1e-4       # minimum improvement to count            [NEW]

WARMUP_RATIO    = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD  = 0.05

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING  [NEW]
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    """
    Monitors val_macro_f1 and signals to stop when it has not improved
    by at least `min_delta` for `patience` consecutive epochs.
    """
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        """Returns True if training should stop."""
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":              "InLegalBERT Encoder",
        "sent_bilstm":       "Sentence BiLSTM",
        "mha_pooling":       "Multi-Head Attn Pooling",
        "ctx_bilstm":        "Context BiLSTM",
        "classifier":        "Classifier Head",
        "crf":               "CRF",
        "dropout":           "Dropout",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({
            "Component":        name,
            "Trainable Params": trainable,
            "Frozen Params":    frozen,
            "Total Params":     trainable + frozen,
        })

    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_all       = total_trainable + total_frozen

    rows.append({
        "Component":        "── TOTAL ──",
        "Trainable Params": total_trainable,
        "Frozen Params":    total_frozen,
        "Total Params":     total_all,
    })

    print("\n" + "=" * 70)
    print("MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF v2)")
    print("=" * 70)
    print(f"  {'Component':<30} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 70)
    for r in rows:
        sep = "─" * 70 if r["Component"] == "── TOTAL ──" else ""
        if sep:
            print(sep)
        print(
            f"  {r['Component']:<30} "
            f"{r['Trainable Params']:>14,} "
            f"{r['Frozen Params']:>10,} "
            f"{r['Total Params']:>12,}"
        )
    print("=" * 70)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}

    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []

        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))

        if not sents or len(sents) != len(labs):
            continue

        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)

        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,   # 128 (output 256)
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,    # 64  (output 128)
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,          # 4
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS, # 8
    ):
        super().__init__()

        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size    # 768
        self.dropout  = nn.Dropout(dropout)

        # ── NEW: Freeze bottom `freeze_layers` encoder layers ─
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2                 # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        ctx_out_dim = ctx_lstm_hidden * 2                   # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim, ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim // 2, num_labels),
        )

        self.crf = CRF(num_tags=num_labels, batch_first=True)

        # Auxiliary CE loss with label smoothing  [NEW]
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        """Freeze embedding + bottom n_freeze encoder layers of BERT."""
        # Always freeze embeddings
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False

        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False

        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1} "
              f"({n_freeze} layers).")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} "
              f"({n_trained} layers) + pooler.\n")

    def encode_sentences(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ) -> torch.Tensor:
        B, T, L = input_ids.shape
        N = B * T

        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0

        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)

        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)

        return sent_vecs.view(B, T, -1)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
    ):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            # CRF loss
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")

            # Auxiliary CE loss with label smoothing  [NEW]
            # emissions: (B, T, C) → (B*T, C); labels: (B, T)
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),        # -100 is ignored by CE
            )

            loss = crf_loss + AUX_CE_WEIGHT * ce_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS HELPER
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    cls_report = classification_report(
        str_trues, str_preds,
        labels=LABELS,
        digits=4,
        zero_division=0,
    )
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1":            macro_f1,
        "micro_f1":            micro_f1,
        "weighted_f1":         weighted_f1,
        "macro_precision":     macro_prec,
        "micro_precision":     micro_prec,
        "weighted_precision":  weighted_prec,
        "macro_recall":        macro_rec,
        "micro_recall":        micro_rec,
        "weighted_recall":     weighted_rec,
        "rare_f1":             rare_f1,
        "rare_precision":      rare_prec,
        "rare_recall":         rare_rec,
        "per_class_metrics":   per_class_metrics,
        "accuracy":            acc,
        "cls_report":          cls_report,
        "cm":                  cm,
        "all_preds":           all_preds,
        "all_trues":           all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        """
        Layer-wise LR decay for BERT encoder  [NEW]:
        - Pooler gets BERT_LR
        - Each encoder layer i (from top=11 to bottom=0) gets
              BERT_LR * (BERT_LR_DECAY ^ (11 - i))
        - Embeddings are frozen, so they are excluded.
        Non-BERT components (head) get HEAD_LR.
        """
        param_groups = []

        # BERT pooler
        param_groups.append({
            "params":       list(self.model.bert.pooler.parameters()),
            "lr":           BERT_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        # BERT encoder layers (top-down decay)
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i          # 0 for top layer
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters()
                      if p.requires_grad]
            if params:
                param_groups.append({
                    "params":       params,
                    "lr":           lr_i,
                    "weight_decay": WEIGHT_DECAY,
                })

        # Head parameters
        head_modules = [
            self.model.sent_bilstm,
            self.model.mha_pooling,
            self.model.sent_layer_norm,
            self.model.ctx_bilstm,
            self.model.classifier,
            self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))

        param_groups.append({
            "params":       head_params,
            "lr":           HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                )
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    n += 1
        return total_loss / max(1, n)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS):

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_DOCS,
            shuffle=True, collate_fn=collate_rrc,
        )
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = warmup_steps,
            num_training_steps = total_steps,
        )

        early_stopper = EarlyStopping()    # [NEW]
        history       = []
        best_f1       = -1.0
        best_state    = None

        total_train_start = time.time()
        actual_epochs     = 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(train_loader):

                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                )

                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1
                    optimizer.zero_grad()
                    continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP
                    )
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            epoch_train_time = time.time() - epoch_start
            avg_train_loss   = running_loss / max(1, n_steps)

            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            nan_info = f" [nan_steps={nan_steps}]" if nan_steps > 0 else ""
            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"val_acc: {val_metrics['accuracy']:.4f} | "
                f"time: {epoch_train_time:.1f}s"
                f" | ES: {early_stopper.counter}/{early_stopper.patience}"
                f"{nan_info}"
            )

            row = {
                "epoch":                   epoch,
                "train_loss":              avg_train_loss,
                "val_loss":                val_loss,
                "val_accuracy":            val_metrics["accuracy"],
                "val_macro_f1":            val_metrics["macro_f1"],
                "val_micro_f1":            val_metrics["micro_f1"],
                "val_weighted_f1":         val_metrics["weighted_f1"],
                "val_rare_f1":             val_metrics["rare_f1"],
                "val_macro_precision":     val_metrics["macro_precision"],
                "val_micro_precision":     val_metrics["micro_precision"],
                "val_weighted_precision":  val_metrics["weighted_precision"],
                "val_rare_precision":      val_metrics["rare_precision"],
                "val_macro_recall":        val_metrics["macro_recall"],
                "val_micro_recall":        val_metrics["micro_recall"],
                "val_weighted_recall":     val_metrics["weighted_recall"],
                "val_rare_recall":         val_metrics["rare_recall"],
                "epoch_train_time_s":      epoch_train_time,
                "nan_steps":               nan_steps,
                "timestamp":               datetime.utcnow().isoformat(),
            }
            history.append(row)

            # Save best checkpoint
            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f} — snapshot saved")

            # Early stopping check  [NEW]
            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping triggered after epoch {epoch} "
                      f"(no improvement for {early_stopper.patience} epochs).\n")
                break

        total_train_time = time.time() - total_train_start
        print(f"\n⏱  Total training time : {total_train_time/60:.2f} min "
              f"({total_train_time:.1f} s)  —  {actual_epochs} epochs run")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        timing_summary = {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / max(1, actual_epochs),
            "num_epochs_run":          actual_epochs,
            "num_epochs_max":          num_epochs,
        }
        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump(timing_summary, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_train_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)

        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)

                decoded, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=None, lengths=lengths,
                )
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    true_seq = labels[i, :true_len].cpu().numpy().tolist()
                    all_preds.extend(seq_preds)
                    all_trues.extend(true_seq)

        if measure_inference_time:
            total_infer_time = time.time() - infer_start
            n_sentences      = len(all_trues)
            infer_info = {
                "split":                      split_name,
                "n_documents":                n_samples,
                "n_sentences":                n_sentences,
                "total_inference_time_s":     total_infer_time,
                "latency_per_document_ms":    total_infer_time / max(1, n_samples) * 1000,
                "latency_per_sentence_ms":    total_infer_time / max(1, n_sentences) * 1000,
                "throughput_sentences_per_s": n_sentences / max(1e-9, total_infer_time),
            }
            with open(
                os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w"
            ) as f:
                json.dump(infer_info, f, indent=2)

            print(f"\n⏱  Inference ({split_name}): "
                  f"{total_infer_time:.2f}s | "
                  f"latency/doc: {infer_info['latency_per_document_ms']:.1f}ms | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        model_args = {
            "bert_model_name":  INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "sent_lstm_layers": SENT_LSTM_LAYERS,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "ctx_lstm_layers":  CTX_LSTM_LAYERS,
            "mha_heads":        MHA_HEADS,
            "mha_dropout":      MHA_DROPOUT,
            "num_labels":       NUM_LABELS,
            "dropout":          DROPOUT,
            "labels":           LABELS,
            "label2id":         label2id,
            "id2label":         id2label,
            "max_seq_length":   MAX_SEQ_LENGTH,
            "rare_threshold":   RARE_THRESHOLD,
            "freeze_layers":    BERT_FREEZE_LAYERS,
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)
        print(f"\n💾 Best model saved to: {BEST_MODEL_DIR}/")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o")
        ax.plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s")
        ax.set_title("Training vs Validation Loss (CRF + CE combined)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_loss_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_micro_f1",    "Micro-F1",    "--"),
            ("val_weighted_f1", "Weighted-F1", "-."),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation F1 Scores"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o")
        axes[1].plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s")
        axes[1].set_title("Loss Curves"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_f1_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_precision",    "Macro-Prec",    "-"),
            ("val_micro_precision",    "Micro-Prec",    "--"),
            ("val_weighted_precision", "Weighted-Prec", "-."),
            ("val_rare_precision",     "Rare-Prec",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation Precision"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        for col, lbl, ls in [
            ("val_macro_recall",    "Macro-Rec",    "-"),
            ("val_micro_recall",    "Micro-Rec",    "--"),
            ("val_weighted_recall", "Weighted-Rec", "-."),
            ("val_rare_recall",     "Rare-Rec",     ":"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[1].set_title("Validation Recall"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_pr_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        if "epoch_train_time_s" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.bar(epochs, hist_df["epoch_train_time_s"], color="steelblue", alpha=0.8)
            ax.axhline(
                hist_df["epoch_train_time_s"].mean(), color="red", linestyle="--",
                label=f"Mean = {hist_df['epoch_train_time_s'].mean():.1f}s"
            )
            ax.set_title("Per-Epoch Training Time")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Time (s)")
            ax.legend(); ax.grid(True, alpha=0.3, axis="y")
            plt.tight_layout()
            p = os.path.join(OUT_DIR, "training_time_curve.png")
            plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d",
                    xticklabels=LABELS, yticklabels=LABELS,
                    cmap="Blues", ax=ax)
        if rare_labels:
            for tick in ax.get_xticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
            for tick in ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        labels = LABELS
        f1s    = [per_class_metrics[l]["f1"] for l in labels]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in labels]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12)
        ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


# ═══════════════════════════════════════════════════════════
# PRINT FULL METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 68)
    print("FINAL RESULTS SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF v2)")
    print("=" * 68)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print("-" * 68)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 68)
    for label, key in rows:
        sep = "─" * 68 if key == "rare_f1" else ""
        if sep:
            print(sep)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 68)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT (top-4 layers fine-tuned) → "
          "Sentence BiLSTM(128) → MHA(4-head) Pooling → "
          "Context BiLSTM(64) → Linear → CRF + Aux-CE")
    print(f"\nAnti-overfitting settings:")
    print(f"  Dropout        : {DROPOUT}  (was 0.3)")
    print(f"  Weight decay   : {WEIGHT_DECAY}  (was 0.01)")
    print(f"  BERT freeze    : bottom {BERT_FREEZE_LAYERS} layers")
    print(f"  BERT LR decay  : {BERT_LR_DECAY} per layer")
    print(f"  Aux CE weight  : {AUX_CE_WEIGHT}  (label_smoothing={LABEL_SMOOTHING})")
    print(f"  Early stopping : patience={ES_PATIENCE}\n")

    print("Loading JSONL files...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    freq_df = pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ])
    freq_df.to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    print("\nInitialising model...")
    model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    )

    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False
    )

    trainer = Trainer(model, device=DEVICE)

    print(f"\nStarting training (max {NUM_EPOCHS} epochs, early stop patience={ES_PATIENCE})...")
    hist_df, total_train_time = trainer.train(
        train_dataset, dev_dataset,
        rare_ids   = rare_ids,
        tokenizer  = tokenizer,
        num_epochs = NUM_EPOCHS,
    )
    print("\nTraining complete.")

    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    print("\nEvaluating on Dev set...")
    dev_metrics = trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                   measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA-Pooling + CRF v2\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    trainer.save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    trainer.save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set...")
    test_metrics = trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                    measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA-Pooling + CRF v2\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    trainer.save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    trainer.save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pred_df = pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    for split, mets in [("dev", dev_metrics), ("test", test_metrics)]:
        rows_pc = []
        for lbl in LABELS:
            pc = mets["per_class_metrics"][lbl]
            rows_pc.append({
                "label":     lbl,
                "is_rare":   lbl in rare_labels,
                "f1":        pc["f1"],
                "precision": pc["precision"],
                "recall":    pc["recall"],
            })
        pd.DataFrame(rows_pc).to_csv(
            os.path.join(OUT_DIR, f"{split}_per_class_metrics.csv"), index=False
        )

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": {
            "name":             "InLegalBERT + BiLSTM + MHA-Pooling + CRF v2",
            "bert_model":       INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "mha_heads":        MHA_HEADS,
            "trainable_params": total_trainable,
            "frozen_params":    total_frozen,
            "total_params":     total_trainable + total_frozen,
        },
        "timing": {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "dev_inference":  dev_metrics.get("inference_time_info", {}),
            "test_inference": test_metrics.get("inference_time_info", {}),
        },
        "rare_classes":   rare_labels,
        "rare_threshold": RARE_THRESHOLD,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        total_train_time = total_train_time,
        total_trainable  = total_trainable,
        total_frozen     = total_frozen,
    )

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT (top-4 layers fine-tuned) → Sentence BiLSTM(128) → MHA(4-head) Pooling → Context BiLSTM(64) → Linear → CRF + Aux-CE

Anti-overfitting settings:
  Dropout        : 0.4  (was 0.3)
  Weight decay   : 0.05  (was 0.01)
  BERT freeze    : bottom 8 layers
  BERT LR decay  : 0.9 per layer
  Aux CE weight  : 0.3  (label_smoothing=0.1)
  Early stopping : patience=10

Loading JSONL files...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED   

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7 (8 layers).
🔥 BERT layers trainable: layers 8-11 (4 layers) + pooler.


MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF v2)
  Component                           Trainable     Frozen        Total
----------------------------------------------------------------------
  InLegalBERT Encoder                28,942,080 80,540,160  109,482,240
  Sentence BiLSTM                     1,314,816          0    1,314,816
  Multi-Head Attn Pooling               197,120          0      197,120
  Context BiLSTM                        264,192          0      264,192
  Classifier Head                         9,101          0        9,101
  CRF                                       195          0          195
  Dropout                                     0          0            0
──────────────────────────────────────────────────────────────────────
  ── TOTAL ──                        30,728,016 80,540,160  111,268,176

Starting training (m

In [3]:
# inlegalbert_bilstm_mha_crf_rrc_v2.py  (ANTI-OVERFITTING REVISION)
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#
# ANTI-OVERFITTING CHANGES vs v1:
#   1. Early stopping (patience=10 on val_macro_f1).
#   2. Freeze bottom 8 of 12 BERT encoder layers; fine-tune only top 4 + pooler.
#   3. Layer-wise LR decay across BERT transformer layers (factor 0.9 per layer).
#   4. Dropout raised from 0.3 → 0.4.
#   5. Weight decay raised from 0.01 → 0.05.
#   6. Sentence BiLSTM hidden reduced 256 → 128 (output 256).
#   7. Context BiLSTM hidden reduced 128 → 64  (output 128).
#   8. MHA heads reduced 8 → 4 (matches new sent_out_dim=256; 256/4=64 ✓).
#   9. Auxiliary cross-entropy loss (label_smoothing=0.1) added alongside CRF
#      to provide a dense gradient signal and regularise the emission head.
#  10. NUM_EPOCHS cap raised to 100 (early stopping will fire much earlier).
#

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_v3_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32          # tokens per sentence
BATCH_DOCS      = 2
NUM_EPOCHS      = 60         # early stopping will fire well before this
BERT_LR         = 1e-5        # CHANGE: 3e-5 → 2e-5; lower layers get further decay
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05        # CHANGE: 0.01 → 0.05 (stronger L2)
GRAD_CLIP       = 1.0
DROPOUT         = 0.4         # CHANGE: 0.3 → 0.4

# ── BERT freeze / layer-wise LR decay ─────────────────────
BERT_FREEZE_LAYERS  = 8       # freeze bottom 8 of 12 encoder layers  [NEW]
BERT_LR_DECAY       = 0.9     # multiply LR by this per layer upward   [NEW]

# ── Sentence-level BiLSTM (token → sentence vector) ───────
SENT_LSTM_HIDDEN = 128        # CHANGE: 256 → 128 (per dir → output 256)
SENT_LSTM_LAYERS = 2

# ── Multi-Head Attention Pooling ──────────────────────────
MHA_HEADS        = 4          # CHANGE: 8 → 4 (256 / 4 = 64 ✓)
MHA_DROPOUT      = 0.1

# ── Context-enrichment BiLSTM (sentence → doc) ────────────
CTX_LSTM_HIDDEN  = 64         # CHANGE: 128 → 64 (per dir → output 128)
CTX_LSTM_LAYERS  = 2

# ── Auxiliary loss ─────────────────────────────────────────
AUX_CE_WEIGHT    = 0.2        # weight for auxiliary CE loss            [NEW]
LABEL_SMOOTHING  = 0.1        # label smoothing for CE                  [NEW]

# ── Early stopping ─────────────────────────────────────────
ES_PATIENCE      = 10         # stop if no improvement for N epochs     [NEW]
ES_MIN_DELTA     = 1e-4       # minimum improvement to count            [NEW]

WARMUP_RATIO    = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD  = 0.05

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING  [NEW]
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    """
    Monitors val_macro_f1 and signals to stop when it has not improved
    by at least `min_delta` for `patience` consecutive epochs.
    """
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        """Returns True if training should stop."""
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":              "InLegalBERT Encoder",
        "sent_bilstm":       "Sentence BiLSTM",
        "mha_pooling":       "Multi-Head Attn Pooling",
        "ctx_bilstm":        "Context BiLSTM",
        "classifier":        "Classifier Head",
        "crf":               "CRF",
        "dropout":           "Dropout",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({
            "Component":        name,
            "Trainable Params": trainable,
            "Frozen Params":    frozen,
            "Total Params":     trainable + frozen,
        })

    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_all       = total_trainable + total_frozen

    rows.append({
        "Component":        "── TOTAL ──",
        "Trainable Params": total_trainable,
        "Frozen Params":    total_frozen,
        "Total Params":     total_all,
    })

    print("\n" + "=" * 70)
    print("MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF v2)")
    print("=" * 70)
    print(f"  {'Component':<30} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 70)
    for r in rows:
        sep = "─" * 70 if r["Component"] == "── TOTAL ──" else ""
        if sep:
            print(sep)
        print(
            f"  {r['Component']:<30} "
            f"{r['Trainable Params']:>14,} "
            f"{r['Frozen Params']:>10,} "
            f"{r['Total Params']:>12,}"
        )
    print("=" * 70)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}

    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []

        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))

        if not sents or len(sents) != len(labs):
            continue

        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)

    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t

    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads

        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)

        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale

        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)

        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)

        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,   # 128 (output 256)
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,    # 64  (output 128)
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,          # 4
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS, # 8
    ):
        super().__init__()

        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size    # 768
        self.dropout  = nn.Dropout(dropout)

        # ── NEW: Freeze bottom `freeze_layers` encoder layers ─
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2                 # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        ctx_out_dim = ctx_lstm_hidden * 2                   # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim, ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim // 2, num_labels),
        )

        self.crf = CRF(num_tags=num_labels, batch_first=True)

        # Auxiliary CE loss with label smoothing  [NEW]
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        """Freeze embedding + bottom n_freeze encoder layers of BERT."""
        # Always freeze embeddings
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False

        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False

        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1} "
              f"({n_freeze} layers).")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} "
              f"({n_trained} layers) + pooler.\n")

    def encode_sentences(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ) -> torch.Tensor:
        B, T, L = input_ids.shape
        N = B * T

        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)

        valid = flat_mask.sum(dim=-1) > 0

        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)

        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)

        token_embs_all = self.dropout(token_embs_all)

        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)

        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False

        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)

        return sent_vecs.view(B, T, -1)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
    ):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0

            # CRF loss
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")

            # Auxiliary CE loss with label smoothing  [NEW]
            # emissions: (B, T, C) → (B*T, C); labels: (B, T)
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),        # -100 is ignored by CE
            )

            loss = crf_loss + AUX_CE_WEIGHT * ce_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS HELPER
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)

    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)

    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    cls_report = classification_report(
        str_trues, str_preds,
        labels=LABELS,
        digits=4,
        zero_division=0,
    )
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1":            macro_f1,
        "micro_f1":            micro_f1,
        "weighted_f1":         weighted_f1,
        "macro_precision":     macro_prec,
        "micro_precision":     micro_prec,
        "weighted_precision":  weighted_prec,
        "macro_recall":        macro_rec,
        "micro_recall":        micro_rec,
        "weighted_recall":     weighted_rec,
        "rare_f1":             rare_f1,
        "rare_precision":      rare_prec,
        "rare_recall":         rare_rec,
        "per_class_metrics":   per_class_metrics,
        "accuracy":            acc,
        "cls_report":          cls_report,
        "cm":                  cm,
        "all_preds":           all_preds,
        "all_trues":           all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        """
        Layer-wise LR decay for BERT encoder  [NEW]:
        - Pooler gets BERT_LR
        - Each encoder layer i (from top=11 to bottom=0) gets
              BERT_LR * (BERT_LR_DECAY ^ (11 - i))
        - Embeddings are frozen, so they are excluded.
        Non-BERT components (head) get HEAD_LR.
        """
        param_groups = []

        # BERT pooler
        param_groups.append({
            "params":       list(self.model.bert.pooler.parameters()),
            "lr":           BERT_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        # BERT encoder layers (top-down decay)
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i          # 0 for top layer
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters()
                      if p.requires_grad]
            if params:
                param_groups.append({
                    "params":       params,
                    "lr":           lr_i,
                    "weight_decay": WEIGHT_DECAY,
                })

        # Head parameters
        head_modules = [
            self.model.sent_bilstm,
            self.model.mha_pooling,
            self.model.sent_layer_norm,
            self.model.ctx_bilstm,
            self.model.classifier,
            self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))

        param_groups.append({
            "params":       head_params,
            "lr":           HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                )
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    n += 1
        return total_loss / max(1, n)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS):

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_DOCS,
            shuffle=True, collate_fn=collate_rrc,
        )
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = warmup_steps,
            num_training_steps = total_steps,
        )

        early_stopper = EarlyStopping()    # [NEW]
        history       = []
        best_f1       = -1.0
        best_state    = None

        total_train_start = time.time()
        actual_epochs     = 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(train_loader):

                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                )

                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1
                    optimizer.zero_grad()
                    continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP
                    )
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            epoch_train_time = time.time() - epoch_start
            avg_train_loss   = running_loss / max(1, n_steps)

            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            nan_info = f" [nan_steps={nan_steps}]" if nan_steps > 0 else ""
            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"train_loss: {avg_train_loss:.4f} | "
                f"val_loss: {val_loss:.4f} | "
                f"val_macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"val_rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"val_acc: {val_metrics['accuracy']:.4f} | "
                f"time: {epoch_train_time:.1f}s"
                f" | ES: {early_stopper.counter}/{early_stopper.patience}"
                f"{nan_info}"
            )

            row = {
                "epoch":                   epoch,
                "train_loss":              avg_train_loss,
                "val_loss":                val_loss,
                "val_accuracy":            val_metrics["accuracy"],
                "val_macro_f1":            val_metrics["macro_f1"],
                "val_micro_f1":            val_metrics["micro_f1"],
                "val_weighted_f1":         val_metrics["weighted_f1"],
                "val_rare_f1":             val_metrics["rare_f1"],
                "val_macro_precision":     val_metrics["macro_precision"],
                "val_micro_precision":     val_metrics["micro_precision"],
                "val_weighted_precision":  val_metrics["weighted_precision"],
                "val_rare_precision":      val_metrics["rare_precision"],
                "val_macro_recall":        val_metrics["macro_recall"],
                "val_micro_recall":        val_metrics["micro_recall"],
                "val_weighted_recall":     val_metrics["weighted_recall"],
                "val_rare_recall":         val_metrics["rare_recall"],
                "epoch_train_time_s":      epoch_train_time,
                "nan_steps":               nan_steps,
                "timestamp":               datetime.utcnow().isoformat(),
            }
            history.append(row)

            # Save best checkpoint
            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f} — snapshot saved")

            # Early stopping check  [NEW]
            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping triggered after epoch {epoch} "
                      f"(no improvement for {early_stopper.patience} epochs).\n")
                break

        total_train_time = time.time() - total_train_start
        print(f"\n⏱  Total training time : {total_train_time/60:.2f} min "
              f"({total_train_time:.1f} s)  —  {actual_epochs} epochs run")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        timing_summary = {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / max(1, actual_epochs),
            "num_epochs_run":          actual_epochs,
            "num_epochs_max":          num_epochs,
        }
        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump(timing_summary, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_train_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)

        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)

                decoded, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=None, lengths=lengths,
                )
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    true_seq = labels[i, :true_len].cpu().numpy().tolist()
                    all_preds.extend(seq_preds)
                    all_trues.extend(true_seq)

        if measure_inference_time:
            total_infer_time = time.time() - infer_start
            n_sentences      = len(all_trues)
            infer_info = {
                "split":                      split_name,
                "n_documents":                n_samples,
                "n_sentences":                n_sentences,
                "total_inference_time_s":     total_infer_time,
                "latency_per_document_ms":    total_infer_time / max(1, n_samples) * 1000,
                "latency_per_sentence_ms":    total_infer_time / max(1, n_sentences) * 1000,
                "throughput_sentences_per_s": n_sentences / max(1e-9, total_infer_time),
            }
            with open(
                os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w"
            ) as f:
                json.dump(infer_info, f, indent=2)

            print(f"\n⏱  Inference ({split_name}): "
                  f"{total_infer_time:.2f}s | "
                  f"latency/doc: {infer_info['latency_per_document_ms']:.1f}ms | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        model_args = {
            "bert_model_name":  INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "sent_lstm_layers": SENT_LSTM_LAYERS,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "ctx_lstm_layers":  CTX_LSTM_LAYERS,
            "mha_heads":        MHA_HEADS,
            "mha_dropout":      MHA_DROPOUT,
            "num_labels":       NUM_LABELS,
            "dropout":          DROPOUT,
            "labels":           LABELS,
            "label2id":         label2id,
            "id2label":         id2label,
            "max_seq_length":   MAX_SEQ_LENGTH,
            "rare_threshold":   RARE_THRESHOLD,
            "freeze_layers":    BERT_FREEZE_LAYERS,
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)
        print(f"\n💾 Best model saved to: {BEST_MODEL_DIR}/")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o")
        ax.plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s")
        ax.set_title("Training vs Validation Loss (CRF + CE combined)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_loss_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_micro_f1",    "Micro-F1",    "--"),
            ("val_weighted_f1", "Weighted-F1", "-."),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation F1 Scores"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o")
        axes[1].plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s")
        axes[1].set_title("Loss Curves"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_f1_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_precision",    "Macro-Prec",    "-"),
            ("val_micro_precision",    "Micro-Prec",    "--"),
            ("val_weighted_precision", "Weighted-Prec", "-."),
            ("val_rare_precision",     "Rare-Prec",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[0].set_title("Validation Precision"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        for col, lbl, ls in [
            ("val_macro_recall",    "Macro-Rec",    "-"),
            ("val_micro_recall",    "Micro-Rec",    "--"),
            ("val_weighted_recall", "Weighted-Rec", "-."),
            ("val_rare_recall",     "Rare-Rec",     ":"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, linestyle=ls, marker="o")
        axes[1].set_title("Validation Recall"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_pr_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        if "epoch_train_time_s" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.bar(epochs, hist_df["epoch_train_time_s"], color="steelblue", alpha=0.8)
            ax.axhline(
                hist_df["epoch_train_time_s"].mean(), color="red", linestyle="--",
                label=f"Mean = {hist_df['epoch_train_time_s'].mean():.1f}s"
            )
            ax.set_title("Per-Epoch Training Time")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Time (s)")
            ax.legend(); ax.grid(True, alpha=0.3, axis="y")
            plt.tight_layout()
            p = os.path.join(OUT_DIR, "training_time_curve.png")
            plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d",
                    xticklabels=LABELS, yticklabels=LABELS,
                    cmap="Blues", ax=ax)
        if rare_labels:
            for tick in ax.get_xticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
            for tick in ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        labels = LABELS
        f1s    = [per_class_metrics[l]["f1"] for l in labels]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in labels]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12)
        ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


# ═══════════════════════════════════════════════════════════
# PRINT FULL METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 68)
    print("FINAL RESULTS SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF v2)")
    print("=" * 68)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print("-" * 68)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 68)
    for label, key in rows:
        sep = "─" * 68 if key == "rare_f1" else ""
        if sep:
            print(sep)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 68)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT (top-4 layers fine-tuned) → "
          "Sentence BiLSTM(128) → MHA(4-head) Pooling → "
          "Context BiLSTM(64) → Linear → CRF + Aux-CE")
    print(f"\nAnti-overfitting settings:")
    print(f"  Dropout        : {DROPOUT}  (was 0.3)")
    print(f"  Weight decay   : {WEIGHT_DECAY}  (was 0.01)")
    print(f"  BERT freeze    : bottom {BERT_FREEZE_LAYERS} layers")
    print(f"  BERT LR decay  : {BERT_LR_DECAY} per layer")
    print(f"  Aux CE weight  : {AUX_CE_WEIGHT}  (label_smoothing={LABEL_SMOOTHING})")
    print(f"  Early stopping : patience={ES_PATIENCE}\n")

    print("Loading JSONL files...")
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)

    freq_df = pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ])
    freq_df.to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)

    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    print("\nInitialising model...")
    model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    )

    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False
    )

    trainer = Trainer(model, device=DEVICE)

    print(f"\nStarting training (max {NUM_EPOCHS} epochs, early stop patience={ES_PATIENCE})...")
    hist_df, total_train_time = trainer.train(
        train_dataset, dev_dataset,
        rare_ids   = rare_ids,
        tokenizer  = tokenizer,
        num_epochs = NUM_EPOCHS,
    )
    print("\nTraining complete.")

    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    print("\nEvaluating on Dev set...")
    dev_metrics = trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                   measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA-Pooling + CRF v2\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    trainer.save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    trainer.save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set...")
    test_metrics = trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                    measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA-Pooling + CRF v2\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    trainer.save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    trainer.save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pred_df = pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    for split, mets in [("dev", dev_metrics), ("test", test_metrics)]:
        rows_pc = []
        for lbl in LABELS:
            pc = mets["per_class_metrics"][lbl]
            rows_pc.append({
                "label":     lbl,
                "is_rare":   lbl in rare_labels,
                "f1":        pc["f1"],
                "precision": pc["precision"],
                "recall":    pc["recall"],
            })
        pd.DataFrame(rows_pc).to_csv(
            os.path.join(OUT_DIR, f"{split}_per_class_metrics.csv"), index=False
        )

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall",
        "accuracy",
    ]
    metrics_summary = {
        "model": {
            "name":             "InLegalBERT + BiLSTM + MHA-Pooling + CRF v2",
            "bert_model":       INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "mha_heads":        MHA_HEADS,
            "trainable_params": total_trainable,
            "frozen_params":    total_frozen,
            "total_params":     total_trainable + total_frozen,
        },
        "timing": {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "dev_inference":  dev_metrics.get("inference_time_info", {}),
            "test_inference": test_metrics.get("inference_time_info", {}),
        },
        "rare_classes":   rare_labels,
        "rare_threshold": RARE_THRESHOLD,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
        "per_class_dev":  dev_metrics["per_class_metrics"],
        "per_class_test": test_metrics["per_class_metrics"],
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        total_train_time = total_train_time,
        total_trainable  = total_trainable,
        total_frozen     = total_frozen,
    )

    print(f"\n📁 All outputs saved to: {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT (top-4 layers fine-tuned) → Sentence BiLSTM(128) → MHA(4-head) Pooling → Context BiLSTM(64) → Linear → CRF + Aux-CE

Anti-overfitting settings:
  Dropout        : 0.4  (was 0.3)
  Weight decay   : 0.05  (was 0.01)
  BERT freeze    : bottom 8 layers
  BERT LR decay  : 0.9 per layer
  Aux CE weight  : 0.2  (label_smoothing=0.1)
  Early stopping : patience=10

Loading JSONL files...
  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED   

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-7 (8 layers).
🔥 BERT layers trainable: layers 8-11 (4 layers) + pooler.


MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF v2)
  Component                           Trainable     Frozen        Total
----------------------------------------------------------------------
  InLegalBERT Encoder                28,942,080 80,540,160  109,482,240
  Sentence BiLSTM                     1,314,816          0    1,314,816
  Multi-Head Attn Pooling               197,120          0      197,120
  Context BiLSTM                        264,192          0      264,192
  Classifier Head                         9,101          0        9,101
  CRF                                       195          0          195
  Dropout                                     0          0            0
──────────────────────────────────────────────────────────────────────
  ── TOTAL ──                        30,728,016 80,540,160  111,268,176

Starting training (m

In [1]:
# inlegalbert_bilstm_mha_crf_rrc_v3.py  (OPTIMAL-FIT REVISION)
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#
# CHANGES vs v2 (targeting train/val curve convergence):
#   1. BERT_FREEZE_LAYERS  : 8  → 10  (freeze more; biggest gap reducer)
#   2. HEAD_LR             : 5e-4 → 2e-4  (head learns slower → less memorization)
#   3. SENT_LSTM_LAYERS    : 2  → 1   (single-layer sufficient; BERT already deep)
#   4. CTX_LSTM_LAYERS     : 2  → 1   (same reason)
#   5. DROPOUT             : 0.4 → 0.5
#   6. WEIGHT_DECAY        : 0.05 → 0.1
#   7. Scheduler           : linear → cosine annealing (smoother, flatter minima)
#   8. GRADIENT_ACCUMULATION_STEPS: 2 → 4 (larger eff. batch, better generalization)
#   9. Stochastic Weight Averaging (SWA) applied in the last 20% of epochs
#  10. AUX_CE_WEIGHT       : 0.2 → 0.3
#  11. ES_PATIENCE         : 10  → 8   (stop before extra overfitting epochs)
#

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup,   # CHANGE: cosine instead of linear
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_v3_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS      = 80          # SWA needs a reasonable ceiling; ES will stop earlier
BERT_LR         = 1e-5
HEAD_LR         = 2e-4        # CHANGE: 5e-4 → 2e-4
WEIGHT_DECAY    = 0.1         # CHANGE: 0.05 → 0.1
GRAD_CLIP       = 1.0
DROPOUT         = 0.5         # CHANGE: 0.4 → 0.5

# ── BERT freeze / layer-wise LR decay ─────────────────────
BERT_FREEZE_LAYERS  = 10      # CHANGE: 8 → 10
BERT_LR_DECAY       = 0.9

# ── Sentence-level BiLSTM ──────────────────────────────────
SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 1          # CHANGE: 2 → 1

# ── Multi-Head Attention Pooling ──────────────────────────
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1

# ── Context-enrichment BiLSTM ─────────────────────────────
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 1          # CHANGE: 2 → 1

# ── Auxiliary loss ─────────────────────────────────────────
AUX_CE_WEIGHT    = 0.3        # CHANGE: 0.2 → 0.3
LABEL_SMOOTHING  = 0.1

# ── Stochastic Weight Averaging  [NEW] ────────────────────
SWA_START_FRAC   = 0.80       # start SWA at 80% of total epochs
SWA_LR           = 5e-5       # constant SWA learning rate

# ── Early stopping ─────────────────────────────────────────
ES_PATIENCE      = 8          # CHANGE: 10 → 8
ES_MIN_DELTA     = 1e-4

# ── Gradient accumulation ──────────────────────────────────
GRADIENT_ACCUMULATION_STEPS = 4   # CHANGE: 2 → 4

WARMUP_RATIO    = 0.05
RARE_THRESHOLD  = 0.05

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":              "InLegalBERT Encoder",
        "sent_bilstm":       "Sentence BiLSTM",
        "mha_pooling":       "Multi-Head Attn Pooling",
        "ctx_bilstm":        "Context BiLSTM",
        "classifier":        "Classifier Head",
        "crf":               "CRF",
        "dropout":           "Dropout",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({
            "Component":        name,
            "Trainable Params": trainable,
            "Frozen Params":    frozen,
            "Total Params":     trainable + frozen,
        })

    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_all       = total_trainable + total_frozen
    rows.append({
        "Component":        "── TOTAL ──",
        "Trainable Params": total_trainable,
        "Frozen Params":    total_frozen,
        "Total Params":     total_all,
    })

    print("\n" + "=" * 70)
    print("MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF v3)")
    print("=" * 70)
    print(f"  {'Component':<30} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 70)
    for r in rows:
        sep = "─" * 70 if r["Component"] == "── TOTAL ──" else ""
        if sep:
            print(sep)
        print(
            f"  {r['Component']:<30} "
            f"{r['Trainable Params']:>14,} "
            f"{r['Frozen Params']:>10,} "
            f"{r['Total Params']:>12,}"
        )
    print("=" * 70)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,   # 1
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,    # 1
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS, # 10
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2   # 256

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        ctx_out_dim = ctx_lstm_hidden * 2     # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim, ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)
        self.ce_loss = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING,
            ignore_index=-100,
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} "
              f"({n_trained} layers) + pooler.\n")

    def encode_sentences(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        lengths:        torch.Tensor = None,
    ) -> torch.Tensor:
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)
        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)
        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels:         torch.Tensor = None,
        lengths:        torch.Tensor = None,
    ):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs = self.dropout(sent_vecs)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)
        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )
            loss = crf_loss + AUX_CE_WEIGHT * ce_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS HELPER
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]

    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)

    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    cls_report = classification_report(str_trues, str_preds, labels=LABELS,
                                       digits=4, zero_division=0)
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)

    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        """Layer-wise LR decay for BERT + separate HEAD_LR for non-BERT."""
        param_groups = []

        # BERT pooler
        param_groups.append({
            "params":       list(self.model.bert.pooler.parameters()),
            "lr":           BERT_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        # BERT encoder layers (top-down decay)
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters()
                      if p.requires_grad]
            if params:
                param_groups.append({
                    "params":       params,
                    "lr":           lr_i,
                    "weight_decay": WEIGHT_DECAY,
                })

        # Head
        head_modules = [
            self.model.sent_bilstm,
            self.model.mha_pooling,
            self.model.sent_layer_norm,
            self.model.ctx_bilstm,
            self.model.classifier,
            self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({
            "params":       head_params,
            "lr":           HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        })

        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset, model_override=None):
        m = model_override if model_override is not None else self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                loss, _ = m(input_ids, attention_mask, token_type_ids,
                            labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    n += 1
        return total_loss / max(1, n)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS):

        train_loader = DataLoader(
            train_dataset, batch_size=BATCH_DOCS,
            shuffle=True, collate_fn=collate_rrc,
        )
        optimizer    = self.build_optimizer()

        # ── Cosine annealing scheduler  [CHANGE] ──────────────
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = warmup_steps,
            num_training_steps = total_steps,
        )

        # ── Stochastic Weight Averaging  [NEW] ────────────────
        swa_model      = AveragedModel(self.model)
        swa_start_ep   = max(1, int(num_epochs * SWA_START_FRAC))
        swa_scheduler  = SWALR(optimizer, swa_lr=SWA_LR,
                               anneal_epochs=5, anneal_strategy="cos")
        swa_active     = False
        print(f"📊 SWA will start at epoch {swa_start_ep}/{num_epochs}.")

        early_stopper = EarlyStopping()
        history       = []
        best_f1       = -1.0
        best_state    = None

        total_train_start = time.time()
        actual_epochs     = 0

        for epoch in range(1, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(train_loader):

                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                loss, _ = self.model(
                    input_ids, attention_mask, token_type_ids,
                    labels=labels, lengths=lengths,
                )
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1
                    optimizer.zero_grad()
                    continue

                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), GRAD_CLIP
                    )
                    optimizer.step()
                    if not swa_active:
                        scheduler.step()
                    optimizer.zero_grad()

                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            # Final partial gradient flush
            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                if not swa_active:
                    scheduler.step()
                optimizer.zero_grad()

            # ── Activate / update SWA ──────────────────────────
            if epoch >= swa_start_ep:
                swa_active = True
                swa_model.update_parameters(self.model)
                swa_scheduler.step()

            epoch_train_time = time.time() - epoch_start
            avg_train_loss   = running_loss / max(1, n_steps)

            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)

            nan_info = f" [nan={nan_steps}]" if nan_steps > 0 else ""
            swa_tag  = " [SWA]" if swa_active else ""
            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"train: {avg_train_loss:.4f} | "
                f"val: {val_loss:.4f} | "
                f"macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"acc: {val_metrics['accuracy']:.4f} | "
                f"ES: {early_stopper.counter}/{early_stopper.patience}"
                f"{swa_tag}{nan_info}"
            )

            row = {
                "epoch":                   epoch,
                "train_loss":              avg_train_loss,
                "val_loss":                val_loss,
                "val_accuracy":            val_metrics["accuracy"],
                "val_macro_f1":            val_metrics["macro_f1"],
                "val_micro_f1":            val_metrics["micro_f1"],
                "val_weighted_f1":         val_metrics["weighted_f1"],
                "val_rare_f1":             val_metrics["rare_f1"],
                "val_macro_precision":     val_metrics["macro_precision"],
                "val_micro_precision":     val_metrics["micro_precision"],
                "val_weighted_precision":  val_metrics["weighted_precision"],
                "val_rare_precision":      val_metrics["rare_precision"],
                "val_macro_recall":        val_metrics["macro_recall"],
                "val_micro_recall":        val_metrics["micro_recall"],
                "val_weighted_recall":     val_metrics["weighted_recall"],
                "val_rare_recall":         val_metrics["rare_recall"],
                "epoch_train_time_s":      epoch_train_time,
                "nan_steps":               nan_steps,
                "swa_active":              swa_active,
                "timestamp":               datetime.utcnow().isoformat(),
            }
            history.append(row)

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n")
                break

        # ── Finalize SWA: update BN statistics ────────────────
        if swa_active:
            print("📐 Updating SWA BatchNorm stats...")
            update_bn(
                DataLoader(train_dataset, batch_size=BATCH_DOCS,
                           shuffle=False, collate_fn=collate_rrc),
                swa_model,
                device=self.device,
            )
            swa_val_loss    = self.compute_val_loss(dev_dataset, model_override=swa_model)
            swa_val_metrics = self.evaluate(dev_dataset, rare_ids,
                                            model_override=swa_model)
            print(f"  SWA val_loss: {swa_val_loss:.4f} | "
                  f"SWA macro_f1: {swa_val_metrics['macro_f1']:.4f}")
            # Use SWA weights if they beat the best individual checkpoint
            if swa_val_metrics["macro_f1"] > best_f1:
                best_f1    = swa_val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in swa_model.module.state_dict().items()}
                print(f"  ✔ SWA weights beat individual checkpoint — using SWA.")

        total_train_time = time.time() - total_train_start
        print(f"\n⏱  Total training time: {total_train_time/60:.2f} min — "
              f"{actual_epochs} epochs")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        timing_summary = {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "avg_epoch_time_s":        total_train_time / max(1, actual_epochs),
            "num_epochs_run":          actual_epochs,
            "swa_start_epoch":         swa_start_ep,
        }
        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump(timing_summary, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_train_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False, model_override=None):
        m = model_override if model_override is not None else self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)
                decoded, _ = m(input_ids, attention_mask, token_type_ids,
                               labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    true_seq = labels[i, :true_len].cpu().numpy().tolist()
                    all_preds.extend(seq_preds)
                    all_trues.extend(true_seq)

        if measure_inference_time:
            total_infer_time = time.time() - infer_start
            n_sentences      = len(all_trues)
            infer_info = {
                "split":                      split_name,
                "n_documents":                n_samples,
                "n_sentences":                n_sentences,
                "total_inference_time_s":     total_infer_time,
                "latency_per_document_ms":    total_infer_time / max(1, n_samples) * 1000,
                "latency_per_sentence_ms":    total_infer_time / max(1, n_sentences) * 1000,
                "throughput_sentences_per_s": n_sentences / max(1e-9, total_infer_time),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
            print(f"\n⏱  Inference ({split_name}): {total_infer_time:.2f}s | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")
        else:
            infer_info = None

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        model_args = {
            "bert_model_name":  INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "sent_lstm_layers": SENT_LSTM_LAYERS,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "ctx_lstm_layers":  CTX_LSTM_LAYERS,
            "mha_heads":        MHA_HEADS,
            "mha_dropout":      MHA_DROPOUT,
            "num_labels":       NUM_LABELS,
            "dropout":          DROPOUT,
            "labels":           LABELS,
            "label2id":         label2id,
            "id2label":         id2label,
            "max_seq_length":   MAX_SEQ_LENGTH,
            "rare_threshold":   RARE_THRESHOLD,
            "freeze_layers":    BERT_FREEZE_LAYERS,
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)
        print(f"\n💾 Best model saved → {BEST_MODEL_DIR}/")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()

        # Mark SWA start on plots if applicable
        swa_ep = None
        if "swa_active" in hist_df.columns:
            swa_rows = hist_df[hist_df["swa_active"] == True]["epoch"]
            if not swa_rows.empty:
                swa_ep = int(swa_rows.iloc[0])

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(epochs, hist_df["train_loss"], label="Train Loss",
                marker="o", markersize=3)
        ax.plot(epochs, hist_df["val_loss"],   label="Val Loss",
                marker="s", markersize=3)
        if swa_ep:
            ax.axvline(swa_ep, color="green", linestyle="--", alpha=0.6,
                       label=f"SWA starts (ep {swa_ep})")
        ax.set_title("Training vs Validation Loss (CRF + CE combined)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_loss_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_micro_f1",    "Micro-F1",    "--"),
            ("val_weighted_f1", "Weighted-F1", "-."),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, markersize=3)
        if swa_ep:
            axes[0].axvline(swa_ep, color="green", linestyle="--", alpha=0.5)
        axes[0].set_title("Validation F1 Scores")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, hist_df["train_loss"], label="Train Loss",
                     marker="o", markersize=3)
        axes[1].plot(epochs, hist_df["val_loss"],   label="Val Loss",
                     marker="s", markersize=3)
        if swa_ep:
            axes[1].axvline(swa_ep, color="green", linestyle="--", alpha=0.5,
                            label=f"SWA starts")
        axes[1].set_title("Loss Curves")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_f1_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_precision",    "Macro-Prec",    "-"),
            ("val_micro_precision",    "Micro-Prec",    "--"),
            ("val_weighted_precision", "Weighted-Prec", "-."),
            ("val_rare_precision",     "Rare-Prec",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls)
        axes[0].set_title("Validation Precision")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)
        for col, lbl, ls in [
            ("val_macro_recall",    "Macro-Rec",    "-"),
            ("val_micro_recall",    "Micro-Rec",    "--"),
            ("val_weighted_recall", "Weighted-Rec", "-."),
            ("val_rare_recall",     "Rare-Rec",     ":"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, linestyle=ls)
        axes[1].set_title("Validation Recall")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_pr_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        if "epoch_train_time_s" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.bar(epochs, hist_df["epoch_train_time_s"],
                   color="steelblue", alpha=0.8)
            ax.axhline(hist_df["epoch_train_time_s"].mean(), color="red",
                       linestyle="--",
                       label=f"Mean = {hist_df['epoch_train_time_s'].mean():.1f}s")
            ax.set_title("Per-Epoch Training Time")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Time (s)")
            ax.legend(); ax.grid(True, alpha=0.3, axis="y")
            plt.tight_layout()
            p = os.path.join(OUT_DIR, "training_time_curve.png")
            plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d",
                    xticklabels=LABELS, yticklabels=LABELS,
                    cmap="Blues", ax=ax)
        if rare_labels:
            for tick in ax.get_xticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
            for tick in ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(
            f"{split_name.capitalize()} Confusion Matrix"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        labels = LABELS
        f1s    = [per_class_metrics[l]["f1"] for l in labels]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in labels]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(labels, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12)
        ax.set_xlabel("F1 Score")
        ax.set_title(
            f"{split_name.capitalize()} Per-Class F1"
            + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else "")
        )
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


# ═══════════════════════════════════════════════════════════
# PRINT FULL METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 68)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA-Pool + CRF v3)")
    print("=" * 68)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print("-" * 68)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 68)
    for label, key in rows:
        sep = "─" * 68 if key == "rare_f1" else ""
        if sep:
            print(sep)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 68)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT (top-2 layers) → "
          "Sentence BiLSTM(128,L=1) → MHA(4-head) → "
          "Context BiLSTM(64,L=1) → Linear → CRF + Aux-CE")
    print(f"\nOptimal-fit settings (v3):")
    print(f"  BERT_FREEZE_LAYERS        : {BERT_FREEZE_LAYERS}  (was 8)")
    print(f"  SENT/CTX LSTM layers      : {SENT_LSTM_LAYERS}/{CTX_LSTM_LAYERS}    (was 2/2)")
    print(f"  HEAD_LR                   : {HEAD_LR}  (was 5e-4)")
    print(f"  DROPOUT                   : {DROPOUT}   (was 0.4)")
    print(f"  WEIGHT_DECAY              : {WEIGHT_DECAY}  (was 0.05)")
    print(f"  Scheduler                 : cosine annealing (was linear)")
    print(f"  GRADIENT_ACCUMULATION     : {GRADIENT_ACCUMULATION_STEPS}  (was 2)")
    print(f"  SWA start                 : {SWA_START_FRAC*100:.0f}% of epochs @ LR={SWA_LR}")
    print(f"  AUX_CE_WEIGHT             : {AUX_CE_WEIGHT}  (was 0.2)")
    print(f"  ES_PATIENCE               : {ES_PATIENCE}  (was 10)\n")

    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    print("Loading tokenizer...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    print("\nInitialising model...")
    model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
    )
    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False
    )

    trainer = Trainer(model, device=DEVICE)
    print(f"\nStarting training (max {NUM_EPOCHS} epochs, ES patience={ES_PATIENCE})...")
    hist_df, total_train_time = trainer.train(
        train_dataset, dev_dataset,
        rare_ids   = rare_ids,
        tokenizer  = tokenizer,
        num_epochs = NUM_EPOCHS,
    )
    print("\nTraining complete.")

    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    print("\nEvaluating on Dev set...")
    dev_metrics = trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                   measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA-Pooling + CRF v3\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    trainer.save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    trainer.save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set...")
    test_metrics = trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                    measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA-Pooling + CRF v3\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    trainer.save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    trainer.save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    for split, mets in [("dev", dev_metrics), ("test", test_metrics)]:
        pd.DataFrame([
            {
                "label":     lbl,
                "is_rare":   lbl in rare_labels,
                "f1":        mets["per_class_metrics"][lbl]["f1"],
                "precision": mets["per_class_metrics"][lbl]["precision"],
                "recall":    mets["per_class_metrics"][lbl]["recall"],
            }
            for lbl in LABELS
        ]).to_csv(os.path.join(OUT_DIR, f"{split}_per_class_metrics.csv"), index=False)

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall","accuracy",
    ]
    metrics_summary = {
        "model": {
            "name":             "InLegalBERT + BiLSTM + MHA-Pooling + CRF v3",
            "bert_model":       INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden": SENT_LSTM_HIDDEN,
            "ctx_lstm_hidden":  CTX_LSTM_HIDDEN,
            "mha_heads":        MHA_HEADS,
            "trainable_params": total_trainable,
            "frozen_params":    total_frozen,
        },
        "timing": {
            "total_training_time_s":   total_train_time,
            "total_training_time_min": total_train_time / 60,
            "dev_inference":  dev_metrics.get("inference_time_info", {}),
            "test_inference": test_metrics.get("inference_time_info", {}),
        },
        "rare_classes":   rare_labels,
        "dev":  {k: dev_metrics[k]  for k in scalar_keys},
        "test": {k: test_metrics[k] for k in scalar_keys},
    }
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        total_train_time = total_train_time,
        total_trainable  = total_trainable,
        total_frozen     = total_frozen,
    )
    print(f"\n📁 All outputs → {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT (top-2 layers) → Sentence BiLSTM(128,L=1) → MHA(4-head) → Context BiLSTM(64,L=1) → Linear → CRF + Aux-CE

Optimal-fit settings (v3):
  BERT_FREEZE_LAYERS        : 10  (was 8)
  SENT/CTX LSTM layers      : 1/1    (was 2/2)
  HEAD_LR                   : 0.0002  (was 5e-4)
  DROPOUT                   : 0.5   (was 0.4)
  WEIGHT_DECAY              : 0.1  (was 0.05)
  Scheduler                 : cosine annealing (was linear)
  GRADIENT_ACCUMULATION     : 4  (was 2)
  SWA start                 : 80% of epochs @ LR=5e-05
  AUX_CE_WEIGHT             : 0.3  (was 0.2)
  ES_PATIENCE               : 8  (was 10)

  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RE

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT layers frozen: embeddings + layers 0-9.
🔥 BERT layers trainable: layers 10-11 (2 layers) + pooler.


MODEL PARAMETER SUMMARY  (InLegalBERT + BiLSTM + MHA-Pool + CRF v3)
  Component                           Trainable     Frozen        Total
----------------------------------------------------------------------
  InLegalBERT Encoder                14,766,336 94,715,904  109,482,240
  Sentence BiLSTM                       919,552          0      919,552
  Multi-Head Attn Pooling               197,120          0      197,120
  Context BiLSTM                        164,864          0      164,864
  Classifier Head                         9,101          0        9,101
  CRF                                       195          0          195
  Dropout                                     0          0            0
──────────────────────────────────────────────────────────────────────
  ── TOTAL ──                        16,057,680 94,715,904  110,773,584

Starting training (max 80 epoc